# Face Anti-Spoofing com PatchNet — Implementação Final (Etapa 3)

**Disciplina de Visão Computacional — Bacharelado em IA — UFG**
Grupo: [preencher integrantes]

Este notebook implementa e avalia, **de ponta a ponta**, a proposta do artigo
*PatchNet: A Simple Face Anti-Spoofing Framework via Fine-Grained Patch Recognition* (CVPR 2022),
sobre um dataset **real de ataques de apresentação** (LCC-FASD).

Estrutura:
1. Setup + download do dataset (LCC-FASD)
2. Imports, semente e detecção do dataset
3. **Experimento A — Baseline**: ResNet-18 (face inteira) + Cross-Entropy
4. **Métricas oficiais de FAS**: EER, APCER, BPCER, ACER (=HTER), ROC, matriz de confusão
5. **Experimento B — Proposta (PatchNet)**: patches de alta resolução + AM-Softmax (margem assimétrica)
6. Generalização de domínio + tabela comparativa Baseline × PatchNet

> Convenção de rótulos (padrão FAS): **1 = Bonafide/Live (real)**, **0 = Spoof (ataque)**.


## 1. Setup do ambiente, Kaggle e download do dataset (LCC-FASD)

In [ ]:
# Execute no Google Colab com GPU (Ambiente de execução > Alterar tipo > GPU T4).
!pip install -q kaggle scikit-learn

from google.colab import files
print("Faça o upload do seu arquivo kaggle.json (Kaggle > Account > Create New API Token):")
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Dataset REAL de ataques de apresentação (impressão + replay), baseado em imagens.
# LCC-FASD é o dataset usado pelo repositório de referência do PatchNet.
!kaggle datasets download -d faber24/lcc-fasd
!unzip -q -o lcc-fasd.zip -d lcc_data
print("Download e extração concluídos.")


## 2. Imports, semente e detecção automática do dataset

In [ ]:
import os, glob, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

# --- Reprodutibilidade ---
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")
LIVE_KEYS = ("real", "live", "genuine", "client", "bonafide")  # tudo o mais = spoof

def _has_images(d):
    try:
        return any(f.lower().endswith(IMG_EXT) for f in os.listdir(d))
    except OSError:
        return False

def _is_class_root(p):
    # Um "class root" e uma pasta cujas subpastas imediatas contem imagens (ex.: .../real, .../spoof)
    subs = [os.path.join(p, d) for d in os.listdir(p) if os.path.isdir(os.path.join(p, d))]
    return len(subs) >= 1 and any(_has_images(s) for s in subs)

def discover_class_roots(base):
    roots = []
    for p in glob.glob(os.path.join(base, "**"), recursive=True):
        if os.path.isdir(p) and _is_class_root(p):
            roots.append(p)
    # remove aninhados redundantes (mantem os mais especificos)
    roots = [r for r in roots if not any((r != o and r.startswith(o + os.sep)) for o in roots)]
    return sorted(roots)

def pick(roots, keys):
    for r in roots:
        if any(k in os.path.basename(r).lower() for k in keys):
            return r
    return None

roots = discover_class_roots("lcc_data")
print("Class roots encontrados:")
for r in roots:
    print("  -", r)

train_dir = pick(roots, ("train",))
test_dir  = pick(roots, ("eval", "test"))
val_dir   = pick(roots, ("develop", "val"))

# Fallbacks robustos
if train_dir is None and roots:
    train_dir = roots[0]
if test_dir is None:
    test_dir = val_dir if val_dir else train_dir

print("\nTREINO   :", train_dir)
print("TESTE    :", test_dir)
print("VALIDacao:", val_dir)
assert train_dir is not None, "Nenhum diretorio de classes encontrado — verifique a estrutura do dataset."


In [ ]:
# Mapeia os rotulos do ImageFolder para a convencao FAS (1=live, 0=spoof),
# independentemente da ordem alfabetica das pastas (real/spoof, Client/Imposter, etc.)
def make_fas_dataset(root, transform):
    ds = datasets.ImageFolder(root, transform=transform)
    live_idx = {i for i, c in enumerate(ds.classes) if any(k in c.lower() for k in LIVE_KEYS)}
    ds.target_transform = (lambda y: 1 if y in live_idx else 0)
    live_names  = [c for i, c in enumerate(ds.classes) if i in live_idx]
    spoof_names = [c for i, c in enumerate(ds.classes) if i not in live_idx]
    print(f"  {root}")
    print(f"    classes={ds.classes} | live={live_names} -> 1 | spoof={spoof_names} -> 0 | total={len(ds)}")
    return ds

def class_balance(ds):
    ys = [ds.target_transform(y) for _, y in ds.samples]
    return {"live(1)": int(np.sum(np.array(ys) == 1)), "spoof(0)": int(np.sum(np.array(ys) == 0))}


## 3. Experimento A — Baseline (ResNet-18, face inteira + Cross-Entropy)

Abordagem tradicional que a proposta deve superar: a face inteira é **redimensionada** para 224×224
(descaracterizando a textura de granulação fina) e classificada com **Cross-Entropy**.

In [ ]:
# Transformacoes do baseline: redimensiona a FACE INTEIRA para 224x224
norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

tf_base_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), norm,
])
tf_base_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(), norm,
])

print("Datasets do baseline:")
base_train_ds = make_fas_dataset(train_dir, tf_base_train)
base_test_ds  = make_fas_dataset(test_dir,  tf_base_eval)
print("Balanceamento treino:", class_balance(base_train_ds))
print("Balanceamento teste :", class_balance(base_test_ds))

BATCH = 64
base_train_loader = torch.utils.data.DataLoader(base_train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
base_test_loader  = torch.utils.data.DataLoader(base_test_ds,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)


In [ ]:
def build_resnet18_classifier():
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, 2)   # 2 classes: spoof(0) / live(1)
    return m.to(DEVICE)

def train_ce(model, loader, epochs=8, lr=1e-4):
    crit = nn.CrossEntropyLoss()
    opt = optim.Adam(model.parameters(), lr=lr)
    hist_loss, hist_acc = [], []
    for ep in range(epochs):
        model.train(); run_loss = 0.0; correct = 0; total = 0
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            out = model(x)
            loss = crit(out, y)
            loss.backward(); opt.step()
            run_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item(); total += x.size(0)
        hist_loss.append(run_loss / total); hist_acc.append(correct / total)
        print(f"[Baseline] Epoch {ep+1}/{epochs} - loss={hist_loss[-1]:.4f} - acc={hist_acc[-1]:.4f}")
    return hist_loss, hist_acc

BASE_EPOCHS = 8
baseline_model = build_resnet18_classifier()
base_loss, base_acc = train_ce(baseline_model, base_train_loader, epochs=BASE_EPOCHS)


In [ ]:
# Curvas de treino do baseline
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(range(1, len(base_loss)+1), base_loss, marker='o', color='red')
plt.title('Baseline — Training Loss'); plt.xlabel('Época'); plt.ylabel('Loss'); plt.grid(alpha=.3)
plt.subplot(1, 2, 2)
plt.plot(range(1, len(base_acc)+1), base_acc, marker='o', color='blue')
plt.title('Baseline — Training Accuracy'); plt.xlabel('Época'); plt.ylabel('Acc'); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig('baseline_curvas.png', dpi=150); plt.show()


## 4. Métricas oficiais de FAS (EER, APCER, BPCER, ACER = HTER)

Diferente da acurácia, a biometria avalia segurança com:
- **EER** (Equal Error Rate): ponto da curva ROC onde FPR ≈ FNR.
- **APCER**: taxa de ataques (spoof) classificados como bonafide = `fp / (tn + fp)`.
- **BPCER**: taxa de bonafide classificados como ataque = `fn / (tp + fn)`.
- **ACER = HTER** = (APCER + BPCER) / 2.

As fórmulas seguem `metrics/metrics_counter.py` do repositório de referência. O *score* usado é a
**probabilidade de Live** produzida por um classificador treinado — não um canal de feature arbitrário.

In [ ]:
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay

def compute_fas_metrics(labels, scores):
    # labels: 1=live, 0=spoof ; scores: probabilidade de LIVE
    labels = np.asarray(labels).astype(int); scores = np.asarray(scores, dtype=float)
    fpr, tpr, thr = roc_curve(labels, scores)          # positivo = live
    fnr = 1 - tpr
    idx = int(np.nanargmin(np.abs(fnr - fpr)))
    eer = float((fpr[idx] + fnr[idx]) / 2.0)
    thr_eer = float(thr[idx])
    preds = (scores >= thr_eer).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    apcer = fp / (tn + fp) if (tn + fp) > 0 else 0.0   # spoof aceito como live
    bpcer = fn / (tp + fn) if (tp + fn) > 0 else 0.0   # live rejeitado como spoof
    acer  = (apcer + bpcer) / 2.0
    acc   = (tp + tn) / max(tp + tn + fp + fn, 1)
    return {"EER": eer, "APCER": apcer, "BPCER": bpcer, "ACER": acer, "HTER": acer,
            "ACC": acc, "AUC": float(auc(fpr, tpr)), "thr": thr_eer,
            "fpr": fpr, "tpr": tpr, "preds": preds, "labels": labels}

def report(name, m):
    print(f"=== {name} ===")
    for k in ("ACC", "AUC", "EER", "APCER", "BPCER", "ACER"):
        print(f"  {k:6s}: {m[k]*100:6.2f}%")

def plot_roc_cm(name, m, fname):
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    ax[0].plot(m["fpr"], m["tpr"], label=f"ROC (AUC={m['AUC']:.3f}, EER={m['EER']*100:.2f}%)")
    ax[0].plot([0, 1], [0, 1], 'k--', alpha=.5)
    ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR'); ax[0].set_title(f'{name} — Curva ROC')
    ax[0].legend(); ax[0].grid(alpha=.3)
    cm = confusion_matrix(m["labels"], m["preds"], labels=[0, 1])
    ConfusionMatrixDisplay(cm, display_labels=['Spoof', 'Live']).plot(ax=ax[1], cmap='Blues', colorbar=False)
    ax[1].set_title(f'{name} — Matriz de Confusão (limiar EER)')
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.show()


In [ ]:
@torch.no_grad()
def scores_ce(model, loader):
    # Score = P(live) = softmax(logits)[:,1]
    model.eval(); ys, ss = [], []
    for x, y in loader:
        p = torch.softmax(model(x.to(DEVICE)), dim=1)[:, 1]
        ss.extend(p.cpu().numpy().tolist()); ys.extend(y.numpy().tolist())
    return ys, ss

y_base, s_base = scores_ce(baseline_model, base_test_loader)
metrics_baseline = compute_fas_metrics(y_base, s_base)
report("Baseline (ResNet-18 face inteira + CE)", metrics_baseline)
plot_roc_cm("Baseline", metrics_baseline, "baseline_roc_cm.png")


## 5. Experimento B — Proposta: PatchNet (patches de granulação fina + PatchLoss)

Versão fiel ao artigo, com quatro elementos:
1. **Patches pequenos de verdade**: redimensionamos para 256 e **recortamos** um patch **128×128** (~25%
   da área), preservando a textura local original (ruído de impressão, Moiré de telas) — não a face inteira.
2. **PatchLoss completa**: dois patches da mesma imagem; uma **perda de similaridade** (MSE) força seus
   descritores a concordarem, e a **AM-Softmax de margem assimétrica** (s=30, m_l=0.4, m_s=0.1) separa as
   classes — conforme `metrics/losses.py::PatchLoss`/`AdMSoftmaxLoss`.
3. **Amostragem balanceada**: corrige o forte desbalanceamento do LCC-FASD (~23:1) no treino.
4. **Agregação multi-patch (TTA)**: na inferência, o *score* de uma imagem é a **média de P(live)** sobre
   K=10 patches — como o PatchNet agrega evidência local.

In [ ]:
# Patches de GRANULACAO FINA (resize 256 -> crop 128) + 2 patches no treino + amostragem balanceada
PATCH_RESIZE, PATCH_SIZE, K_TTA = 256, 128, 10

tf_patch = transforms.Compose([
    transforms.Resize((PATCH_RESIZE, PATCH_RESIZE)),
    transforms.RandomCrop(PATCH_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), norm,
])

def make_fas_base(root):
    # ImageFolder SEM transform (retorna PIL) com mapeamento FAS (1=live, 0=spoof)
    ds = datasets.ImageFolder(root)
    live_idx = {i for i, c in enumerate(ds.classes) if any(k in c.lower() for k in LIVE_KEYS)}
    ds.target_transform = (lambda y: 1 if y in live_idx else 0)
    return ds

class TwoPatchDataset(torch.utils.data.Dataset):
    # Treino: dois patches independentes da MESMA imagem (consistencia do PatchNet)
    def __init__(self, base, tf): self.base, self.tf = base, tf
    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        img, y = self.base[i]
        return self.tf(img), self.tf(img), y

class MultiPatchDataset(torch.utils.data.Dataset):
    # Avaliacao: K patches por imagem para agregacao (TTA)
    def __init__(self, base, tf, k=K_TTA): self.base, self.tf, self.k = base, tf, k
    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        img, y = self.base[i]
        return torch.stack([self.tf(img) for _ in range(self.k)]), y

print("Datasets do PatchNet (patches finos):")
patch_train_base = make_fas_base(train_dir)
patch_test_base  = make_fas_base(test_dir)
train_labels = np.array([patch_train_base.target_transform(y) for _, y in patch_train_base.samples])
print(f"  treino: {len(train_labels)} imgs | live={int((train_labels==1).sum())} spoof={int((train_labels==0).sum())}")

# Amostragem balanceada (corrige o ~23:1): peso inversamente proporcional a frequencia da classe
class_count = np.bincount(train_labels, minlength=2)
sample_w = (1.0 / np.clip(class_count, 1, None))[train_labels]
sampler = torch.utils.data.WeightedRandomSampler(torch.as_tensor(sample_w, dtype=torch.double), num_samples=len(train_labels), replacement=True)

patch_train_loader = torch.utils.data.DataLoader(TwoPatchDataset(patch_train_base, tf_patch), batch_size=BATCH, sampler=sampler, num_workers=2, pin_memory=True)
patch_test_loader  = torch.utils.data.DataLoader(MultiPatchDataset(patch_test_base, tf_patch), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)


In [ ]:
# AM-Softmax (Additive Margin) com margens assimetricas — adaptado de fas-patchnet/metrics/losses.py
class AMSoftmaxLoss(nn.Module):
    def __init__(self, in_features, out_features=2, s=30.0, m_l=0.4, m_s=0.1):
        super().__init__()
        self.s = s
        self.m = [m_s, m_l]                      # indexado pelo rotulo: 0=spoof->m_s, 1=live->m_l
        self.fc = nn.Linear(in_features, out_features, bias=False)
        nn.init.xavier_uniform_(self.fc.weight)

    def cosine(self, x):
        x = F.normalize(x, dim=1)
        w = F.normalize(self.fc.weight, dim=1)
        return F.linear(x, w)                    # cos(theta) entre features e protótipos de classe

    def forward(self, x, labels):
        labels = labels.long()
        cos = self.cosine(x)
        m = torch.tensor([self.m[int(l)] for l in labels], device=x.device)
        numerator = self.s * (cos[torch.arange(cos.size(0)), labels] - m)
        excl = torch.cat([torch.cat((cos[i, :y], cos[i, y+1:])).unsqueeze(0) for i, y in enumerate(labels)], dim=0)
        denom = torch.exp(numerator) + torch.sum(torch.exp(self.s * excl), dim=1)
        return -torch.mean(numerator - torch.log(denom))



# PatchLoss completa: similaridade entre 2 patches + AM-Softmax em ambos (fas-patchnet/metrics/losses.py)
class PatchLoss(nn.Module):
    def __init__(self, descriptor_size=256, s=30.0, m_l=0.4, m_s=0.1, alpha1=1.0, alpha2=1.0):
        super().__init__()
        self.amsm = AMSoftmaxLoss(descriptor_size, 2, s=s, m_l=m_l, m_s=m_s)
        self.alpha1, self.alpha2 = alpha1, alpha2

    def forward(self, d1, d2, label):
        l1 = self.amsm(d1, label)
        l2 = self.amsm(d2, label)
        sim = F.mse_loss(F.normalize(d1, dim=1), F.normalize(d2, dim=1))   # consistencia entre patches
        return self.alpha1 * sim + self.alpha2 * (l1 + l2)


class PatchNet(nn.Module):
    # ResNet-18 (backbone) -> descritor -> PatchLoss (similaridade + AM-Softmax)
    def __init__(self, descriptor_size=256, s=30.0, m_l=0.4, m_s=0.1):
        super().__init__()
        bb = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        in_ftrs = bb.fc.in_features              # 512
        bb.fc = nn.Identity()
        self.backbone = bb
        self.descriptor = nn.Sequential(nn.Linear(in_ftrs, descriptor_size), nn.BatchNorm1d(descriptor_size))
        self.patch_loss = PatchLoss(descriptor_size, s=s, m_l=m_l, m_s=m_s)

    def features(self, x):
        return self.descriptor(self.backbone(x))

    def forward(self, x1, x2, label):            # treino -> loss (2 patches)
        return self.patch_loss(self.features(x1), self.features(x2), label)

    @torch.no_grad()
    def live_prob(self, x):                       # inferencia -> P(live)
        cos = self.patch_loss.amsm.cosine(self.features(x))
        return torch.softmax(self.patch_loss.amsm.s * cos, dim=1)[:, 1]


In [ ]:
def train_patchnet(model, loader, epochs=12, lr=1e-4):
    opt = optim.Adam(model.parameters(), lr=lr)
    hist = []
    for ep in range(epochs):
        model.train(); run = 0.0; n = 0
        for x1, x2, y in loader:
            x1, x2, y = x1.to(DEVICE), x2.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = model(x1, x2, y)
            loss.backward(); opt.step()
            run += loss.item() * x1.size(0); n += x1.size(0)
        hist.append(run / n)
        print(f"[PatchNet] Epoch {ep+1}/{epochs} - PatchLoss={hist[-1]:.4f}")
    return hist

PATCH_EPOCHS = 12
patchnet_model = PatchNet().to(DEVICE)
patch_loss_hist = train_patchnet(patchnet_model, patch_train_loader, epochs=PATCH_EPOCHS)

plt.figure(figsize=(6, 4))
plt.plot(range(1, len(patch_loss_hist)+1), patch_loss_hist, marker='o', color='green')
plt.title('PatchNet — PatchLoss (similaridade + AM-Softmax)'); plt.xlabel('Época'); plt.ylabel('Loss'); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig('patchnet_curva_loss.png', dpi=150); plt.show()


In [ ]:
@torch.no_grad()
def scores_patchnet(model, loader):
    # Agregacao multi-patch (TTA): media de P(live) sobre os K patches de cada imagem
    model.eval(); ys, ss = [], []
    for patches, y in loader:                 # patches: (B, K, 3, H, W)
        B, K = patches.shape[0], patches.shape[1]
        flat = patches.view(B * K, *patches.shape[2:]).to(DEVICE)
        p = model.live_prob(flat).view(B, K).mean(dim=1)
        ss.extend(p.cpu().numpy().tolist()); ys.extend(y.numpy().tolist())
    return ys, ss

y_patch, s_patch = scores_patchnet(patchnet_model, patch_test_loader)
metrics_patchnet = compute_fas_metrics(y_patch, s_patch)
report("PatchNet (patches finos + PatchLoss + TTA)", metrics_patchnet)
plot_roc_cm("PatchNet", metrics_patchnet, "patchnet_roc_cm.png")


## 6. Generalização de domínio e comparação final

Ambos os modelos foram **treinados no split de treino e avaliados no split de evaluation** do
LCC-FASD (condições de captura diferentes) — um teste real de generalização. A tabela abaixo
consolida as métricas para a discussão do artigo.

> Reporte os números **como saíram**. Se a proposta não superar o baseline em alguma métrica,
> isso é um resultado real a ser analisado (não há fabricação).

In [ ]:
import pandas as pd

def row(m):
    return {k: round(m[k]*100, 2) for k in ("ACC", "AUC", "EER", "APCER", "BPCER", "ACER")}

comp = pd.DataFrame({
    "Baseline (face inteira + CE)": row(metrics_baseline),
    "PatchNet (patches finos + PatchLoss + TTA)": row(metrics_patchnet),
}).T
comp.columns = [c + " (%)" for c in comp.columns]
print("Comparação Baseline × PatchNet (LCC-FASD, split de evaluation):")
print(comp.to_string())
comp.to_csv("comparacao_metricas.csv")

# Melhor modelo por métrica de segurança (menor é melhor para EER/ACER)
print("\nMenor EER :", comp["EER (%)"].idxmin())
print("Menor ACER:", comp["ACER (%)"].idxmin())


---
### Observações de honestidade científica (para o artigo)
- O **LCC-FASD** contém ataques de apresentação reais (impressão/replay), portanto a premissa do
  PatchNet (textura local) é aplicável — diferente do dataset 140k (faces GAN) usado na Etapa 2.
- Todas as métricas vêm de `compute_fas_metrics` sobre scores `P(live)` de classificadores
  **treinados**; nenhum número é estimado ou fixado manualmente.
- Figuras salvas: `baseline_curvas.png`, `baseline_roc_cm.png`, `patchnet_curva_loss.png`,
  `patchnet_roc_cm.png`, e a tabela `comparacao_metricas.csv` — anexar ao artigo/slides.
